In [73]:
import sys
import os
import gc

# This adds the parent directory (..) to the Python path
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

import logging
from datasets import load_dataset
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW

from dataclasses import dataclass, field

from typing import List, Optional, Tuple, Any, Dict

from transformers import (
    PreTrainedModel,
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    TrainerCallback,
    Trainer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_linear_schedule_with_warmup
)

from transformers.modeling_outputs import ModelOutput

DATA_DIR = Path("../data/")
MODELS_DIR = Path("../models/")
pos_list = ['noun',
            'adverb',
            'adjective',
            'verb',
            'preposition',
            'misc',
            'number',
            'not-no',
            'determiner']

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import root_mean_squared_error, accuracy_score
from scipy.stats import pearsonr
from wordfreq import zipf_frequency


def merge_cols(batch, cols_to_merge, sep_token):
    """
    Merge specified item text components into a single input string per example.

    Designed for use with Hugging Face `Dataset.map` in batched mode.

    Args:
        batch (dict): Batched dataset examples.
        cols_to_merge (list[str]): Ordered column names to concatenate.
        sep_token (str): Separator string between components.

    Returns:
        dict: Dictionary with key `"input_text"` containing merged text strings.
    """
    rows=zip(*(batch[col] for col in cols_to_merge))
    return {
        "input_text": [
            sep_token.join(str(x).strip() for x in row)
            for row in rows
        ]
    }
    
def preprocess_dataset(ds_dict, cols_to_merge, sep_token):
    """
    Preprocess a Hugging Face DatasetDict by:
        1. Merging specified text columns into a single 'input_text' column.
        2. Renaming the target column 'GLMM_score' to 'label'.
        3. Removing all other columns except 'input_text', 'l1' and 'label'.

    Args:
        ds_dict (datasets.DatasetDict): Input DatasetDict with one or more splits.
        cols_to_merge (list[str]): List of text columns to concatenate into 'input_text'.
        sep_token (str): Separator string to insert between merged columns.

    Returns:
        datasets.DatasetDict: Preprocessed DatasetDict where each split contains only
            the columns 'input_text' and 'label', ready for tokenization.
    """
    
    # Compute columns to remove
    first_split = next(iter(ds_dict.values())) # get first key in ds_dict
    all_columns = first_split.column_names
    cols_to_keep = ["input_text", "L1", "en_target_pos"]
    cols_to_remove = [c for c in all_columns if c not in cols_to_keep and c != 'GLMM_score']
    
    # Format input text, rename target label and remove extra columns
    ds_dict = ds_dict.map(
        merge_cols,
        batched=True,
        fn_kwargs={"cols_to_merge": cols_to_merge, "sep_token": sep_token},
        desc="Formatting input text"
    )
    
    # Compute Zipf frequency of english target word
    ds_dict = ds_dict.map(
        batch_compute_frequency,
        batched=True,
        fn_kwargs={"col":"en_target_word"},
        desc="compute zipf frequency of en target word"
    )
    
    # Rename target columns and remove extra columns
    # final target label lists: ['glmm_label', 'freq_label', 'pos_label']
    ds_dict = ds_dict.rename_column("GLMM_score", "labels")
    ds_dict = ds_dict.remove_columns(cols_to_remove)
    
    return ds_dict
    
def batch_compute_frequency(batch, col='en_target_word'):
    """
    zipf_freq: Zipf frequency of a given word based on corpus used by wordfreq
    """
    rows = batch[col]
    return {
            "freq_feature": [zipf_frequency(row.lower(), lang='en') for row in rows]
            }

def cleanup_trainer_memory(*objects):
    """
    Free up memory used by PyTorch and delete given objects.

    Args:
        *objects: Any Python objects (e.g., Trainer, datasets) to delete.
    """
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    for obj in objects:
        del obj

def load_data_paths(data_dir, l1, mode, dataset_split=None):
    """
    Loads CSV file paths for Hugging Face `load_dataset`.

    Returns a dictionary with keys 'train', 'validation', and 'test',
    mapping to lists of CSV paths. Keys with no files are omitted.

    Args:
        data_dir (Path): Base directory containing the data splits.
        l1 (str): Language code (e.g., 'es') or 'xx' to include all languages during finetune.
        mode (str): One of 'finetune', 'predict', or 'evaluate'.
        dataset_split (str or None): For predict/evaluate, which data splits to load:
            'dev', 'test', or 'both'. Ignored for finetune.

    Returns:
        dict: Mapping HF split names to lists of CSV paths, e.g.,
            {'train': ['train/es/file1.csv'], 'validation': ['dev/es/file1.csv']}
    """

    # for finetuning 'xx' means combine all L1s
    # L1s are separate for predict and evaluate
    l1_list = ["es", "de", "cn"] if mode == "finetune" and l1 == "xx" else [l1]

    data_files = {}

    # Mapping from KVL dataset split names to HF split names
    kvl_splits_to_hf = {
        "train": "train",
        "dev": "validation",
        "test": "test"
    }

    if mode == "finetune":
        train_files = []
        val_files = []

        for l1 in l1_list:
            train_folder=data_dir / "train" / l1
            val_folder=data_dir / "dev" / l1

            train_files.extend(str(f) for f in train_folder.glob("*.csv"))
            val_files.extend(str(f) for f in val_folder.glob("*.csv"))

        if train_files:
            data_files["train"] = train_files
        if val_files:
            data_files["validation"] = val_files

    elif mode in {"predict", "evaluate"}:
        
        splits_to_load = ["dev", "test"] if dataset_split == "both" else [dataset_split]

        for folder_name in splits_to_load:
            files = []
            for l1 in l1_list:
                folder = data_dir / folder_name / l1
                files.extend(str(f) for f in folder.glob("*.csv"))

            hf_key = kvl_splits_to_hf[folder_name]
            if files:
                data_files[hf_key] = files

    return data_files

def compute_metrics(eval_pred, debug=False):
    predictions, label_ids = eval_pred
    
    metrics = {}
    
    if debug:
        # --- DEBUGGING PRINTS (Will show up above your progress bar) ---
        print(f"\n[EVAL DEBUG] Predictions type: {type(predictions)}")
        if isinstance(predictions, tuple):
            print(f"[EVAL DEBUG] Predictions tuple length: {len(predictions)}")
            
        print(f"[EVAL DEBUG] Label IDs type: {type(label_ids)}")
        if isinstance(label_ids, tuple):
            print(f"[EVAL DEBUG] Label IDs tuple length: {len(label_ids)}")
        # -------------------------------------------------------------

    # 1. Primary Task (Difficulty)
    logits = predictions[0] if isinstance(predictions, tuple) else predictions
    labels = label_ids[0] if isinstance(label_ids, tuple) else label_ids
    
    logits = logits.flatten()
    labels = labels.flatten()
    
    metrics['rmse'] = root_mean_squared_error(labels, logits)
    metrics['pearson'] = pearsonr(logits, labels)[0]
    
    # 2. Auxiliary Task (POS tagging)
    # If the Trainer correctly passed both outputs and both labels...
    if isinstance(predictions, tuple) and len(predictions) > 1:
        if isinstance(label_ids, tuple) and len(label_ids) > 1:
            
            pos_logits = predictions[1]
            pos_labels = label_ids[1]
            
            # Ensure they aren't None before calculating
            if pos_logits is not None and pos_labels is not None:
                pos_logits = pos_logits
                pos_labels = pos_labels.flatten()
                pos_preds = np.argmax(pos_logits, axis=-1)
                metrics["pos_acc"] = accuracy_score(pos_labels, pos_preds)
            else:
                print("[EVAL DEBUG] Freq Logits or Freq Labels are None!")
        else:
            print("[EVAL DEBUG] label_ids is not a tuple. Did you set label_names?")
            
    return metrics

In [75]:
class OptimizerStateMonitor(TrainerCallback):
    """
    A custom Hugging Face callback to inspect optimizer groups, learning rates, 
    and the real-time values/gradients of specific custom parameters.
    """
    def __init__(self, params_to_watch: list):
        # Pass in strings that match the parameter names you want to track
        self.params_to_watch = params_to_watch

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Triggers exactly when the Trainer prints the training loss to the console."""
        
        optimizer = kwargs.get("optimizer")
        model = kwargs.get("model")

        print(f"\n{'-'*60}")
        print(f"🔍 OPTIMIZER & PARAMETER STATE | Step: {state.global_step}")
        print(f"{'-'*60}")
        
        # Map Parameter Memory IDs to their Optimizer Group Index
        param_to_group_idx = {}

        # 1. Inspect Optimizer Parameter Groups
        if optimizer is not None:
            print("Optimizers (Learning Rates & Weight Decay):")
            for i, group in enumerate(optimizer.param_groups):
                lr = group.get('lr', 0.0)
                weight_decay = group.get('weight_decay', 0.0)
                # Count how many parameter tensors are in this group
                num_params = len(group.get('params', []))
                print(f"  [Group {i}] Params: {num_params} | LR: {lr:.2e} | L2 Decay: {weight_decay}")
                
                # Store the group index for every parameter object in this group
                for p in group.get('params', []):
                    param_to_group_idx[id(p)] = i
        else:
            print("  [Warning] Optimizer not found in kwargs.")

        # 2. Inspect Custom Parameter Values and Live Gradients
        if model is not None:
            
            print("\nTracked Parameters (Values & Gradients):")
            for name, param in model.named_parameters():
                # Check if this parameter name contains any of our target strings
                if any(target in name for target in self.params_to_watch):
                    
                    # Safely detach and convert to numpy for clean printing
                    val = param.data.detach().cpu().numpy()
                    
                    # Check if gradients have been computed yet
                    if param.grad is not None:
                        grad = param.grad.data.detach().cpu().numpy()
                    else:
                        grad = "No Gradient Yet"
                    
                    # Look up the group using the parameter's unique memory ID
                    group_idx = param_to_group_idx.get(id(param), "Unassigned")

                    # Format the output cleanly
                    print(f"  -> {name} [Group {group_idx}]:")
                    print(f"       Value : {np.array2string(val, precision=6, separator=', ')}")
                    print(f"       Grad  : {np.array2string(grad, precision=6, separator=', ') if isinstance(grad, np.ndarray) else grad}")
        print(f"{'-'*60}\n")

In [76]:
@dataclass
class CustomClassifierOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None

@dataclass
class CustomDataCollator(DataCollatorWithPadding):
    custom_features: List[str] = field(default_factory=lambda: ["freq_feature", "pos_label", "labels"])

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        extracted_features = {key: [] for key in self.custom_features}
        
        for feature in features:
            for key in self.custom_features:
                if key in feature:
                    extracted_features[key].append(feature.pop(key))

        batch = super().__call__(features)

        if "pos_label" in extracted_features and extracted_features["pos_label"]:
            batch["pos_label"] = torch.tensor(extracted_features["pos_label"], dtype=torch.long)
        
        if "freq_feature" in extracted_features and extracted_features['freq_feature']:
            batch['freq_feature'] = torch.tensor(extracted_features['freq_feature'], dtype=torch.float32)
        
        if "labels" in extracted_features and extracted_features["labels"]:
            batch["labels"] = torch.tensor(extracted_features["labels"], dtype=torch.float32)
            
        return batch

In [77]:
class Rational(nn.Module):
    """
    Rational Activation Function (Version A)
    Translated natively for PyTorch 2.1.0+
    
    Formula: F(x) = P(x) / (1 + |Q(x)|)
    P(x) = a_0 + a_1*x + a_2*x^2 + ... + a_m*x^m
    Q(x) =       b_1*x + b_2*x^2 + ... + b_n*x^n
    """
    def __init__(self, approx_func="leaky_relu", degrees=(5, 4), trainable=True):
        super(Rational, self).__init__()
        self.degrees = degrees
        
        # The official library initializes the weights to approximate standard functions.
        # Here are the exact pre-computed coefficients to approximate Leaky ReLU.
        if approx_func == "leaky_relu" and degrees == (5, 4):
            init_num = [0.0274, 0.4907, 0.0538, 0.1655, 0.0272, 0.0381]
            init_den = [         0.9329, 0.0210, 0.3341, 0.0221]
        elif approx_func == "identity":
            # Fallback: F(x) = x
            init_num = [0.0] * (degrees[0] + 1)
            init_num[1] = 1.0
            init_den = [0.0] * degrees[1]
        else:
            raise ValueError("Only 'leaky_relu' or 'identity' initializations are provided in this script.")

        # Register parameters
        self.numerator = nn.Parameter(torch.tensor(init_num, dtype=torch.float32), requires_grad=trainable)
        self.denominator = nn.Parameter(torch.tensor(init_den, dtype=torch.float32), requires_grad=trainable)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Evaluate Numerator P(x) using Horner's Method
        num = self.numerator[-1]
        for i in range(self.degrees[0] - 1, -1, -1):
            num = self.numerator[i] + x * num
            
        # 2. Evaluate Denominator Q(x) using Horner's Method
        den = self.denominator[-1]
        for i in range(self.degrees[1] - 2, -1, -1):
            den = self.denominator[i] + x * den
        
        # Multiply by x one last time because Q(x) has no b_0 constant term
        den = x * den 
        
        # 3. Apply Version A Bounds (1 + |Q(x)|)
        den = 1.0 + torch.abs(den)
        
        # 4. Final Rational Output
        return num / den

In [78]:
class ScalarMix(nn.Module):
    def __init__(
        self, 
        mixture_size: int = 13, 
        temperature: float = 2.0,       # Enforces a smooth distribution
        layer_dropout: float = 0.1      # Forces the network to use multiple layers
    ) -> None:
        """
        A stabilized Layer Mixer designed for Multi-Task architectures.
        Prevents Softmax Saturation and Pre-Norm variance explosions.
        """
        super().__init__()
        self.mixture_size = mixture_size
        # self.temperature = temperature
        # Initialize uniformly. With temperature, this starts as a perfectly flat distribution.
        self.scalar_weights = nn.Parameter(torch.zeros(mixture_size))
        self.gamma = nn.Parameter(torch.tensor([1.0]))
        # self.register_buffer('gamma', torch.tensor([1.0]))

    def forward(
        self, 
        tensors: List[torch.Tensor],
        mask: torch.bool = None
    ) -> torch.Tensor:
        
        # Locks raw weights between -1.0 and 1.0. Impossible to explode.
        # bounded_weights = torch.tanh(self.scalar_weights)
        
        # 1. Calculate softmax(w_j)
        normed_weights = F.softmax(self.scalar_weights, dim=0)

        # 2. Stack the transformer layer outputs
        # Shape: (num_layers, batch_size, seq_len, hidden_size)
        tensors_stacked = torch.stack(tensors, dim=0)
        
        # 3. Broadcast weights to match the tensor dimensions
        weights_shape = (-1,) + (1,) * (tensors_stacked.dim() - 1)
        normed_weights = normed_weights.view(weights_shape)
        
        # 4. Calculate sum(softmax(w_j) * t_j)
        mixed_tensor = torch.sum(normed_weights * tensors_stacked, dim=0)
        
        # mixed_tensor = torch.clamp(mixed_tensor, min = 1e-9, max = 10)
        
        # 5. Apply gamma multiplier
        # Optional: Mask out padding tokens for the downstream pooler
        if mask is not None:
            broadcast_mask = mask.unsqueeze(-1)
            mixed_tensor = mixed_tensor * broadcast_mask


        return self.gamma * mixed_tensor

In [79]:
class Backbone(PreTrainedModel):
    config_class = AutoConfig
    def __init__(
        self,
        config
    ):
        """
        Instantiate the model backbone as a subclass of PreTrainedModel.
        Due to transformer's design pattern, we have to assign model prefix for backbones of different structures. 
        e.g. roberta: ["xlm-roberta", "roberta"]
        """
        super().__init__(config)
        self.config = config
        model_type = config.model_type
        if model_type in ['xlm-roberta', 'roberta']:
            # add_pooling_layer=False to get raw hidden states
            self.roberta = AutoModel.from_config(config, add_pooling_layer=False)
            self.__class__.base_model_prefix = "roberta"
    
    def forward(self):
        raise NotImplementedError("ERROR: forward not implemented!")

class CloseTrack_Predictor(Backbone):
    def __init__(
        self,
        config,
        dropout: float = 0.1,
        layer_wise: Optional[str] = None,
        token_agg: Optional[str] = None,
        pred_head: str = 'mlp',
        temp: float = 2.0,
        **kwargs
    ):
        """
        Our customized predictor that performs layer fusion and/or token fusion strategies to exploit transformer's internal structure. Currently, we have implemented 
        one method for layer fusion (layer_wise) and two for token fusion (token_wise). If both are not provided, the predictor reduces to the ordinary regressor, 
        using the first token from the last hidden state as predictor. 
            - layer_wise:
                - "ScalarMix": the layer fusion method that learns a global set of weights for each internal layers before pooling into a single hidden state. 
            - token_wise: 
                - 'AddAttn': the Additive Attention (see [https://d2l.ai/chapter_attention-mechanisms-and-transformers/attention-scoring-functions.html#additive-attention])
                - 'SelfAttn': the Multi-head self-attention, implemented by the PyTorch off-the-self nn.MultiheadAttention function.
                    Since the Positional Encoding are used in transformer training, two types of positional encoder are used for self-attention for investigation
                    - Sinusodial: [https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#positional-encoding]
                    - Learned: [https://machinelearningmastery.com/positional-encodings-in-transformer-models/]
            - pred_head: the regressor used for prediction
                - 'mlp': the same MLP regressor as used in XLMRobertaForSequenceClassification
                - 'vib': a testing regressor that follows the idea of Variational Information Bottleneck, which is essentially a regularized regressor.
        """
        super().__init__(config)

        # initiate layer fusion strategy
        self.layer_wise = layer_wise
        if self.layer_wise == 'ScalarMix':
            self.layer_agg = ScalarMix(
                config.num_hidden_layers + 1,
                temperature=temp,
                layer_dropout=dropout
                )

        self.token_agg = token_agg

        # initiate regressor
        self.pred_head = pred_head
        if self.pred_head == 'mlp':
            self.regressor = nn.Sequential(
                nn.Linear(config.hidden_size, config.hidden_size),
                Rational(approx_func="leaky_relu", degrees=(5, 4)),
                nn.Dropout(dropout),
                nn.Linear(config.hidden_size, config.num_labels)
            )

        # Initialize weights (Hugging Face standard)
        self.post_init()

    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None,        
        freq_feature=None,
        pos_label=None,
        **kwargs
        ) -> CustomClassifierOutput:
        """
        Argument:
            input_ids: the tokenized input 
            attention_mask: the sequence mask output by the tokenizer
            labels: the target labels
        Return:
            CustomClassifierOutput(
                loss: prediction loss
                logits: predicted labels
                hidden_states: all hidden states generated by the backbone
                attentions: all attention scores calculated by the backbone
                token_attn_weights: the token-level attention weights calculated by the token fusion method.
            )
        """

        # The Trainer passes 'num_items_in_batch' which the backbone doesn't accept.
        if "num_items_in_batch" not in self.config:
            kwargs.pop("num_items_in_batch", None)

        # identify the backbone and generate the hidden states. 
        if hasattr(self, "roberta"):
            outputs = self.roberta(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                **kwargs
            )

        # if layer fusion is applied
        if self.layer_wise:
            # mixes the entire sequences from all layers, not just the first token to support following token-wise aggregation
            hiddens = list(outputs.hidden_states) # [layer, batch, seq, hidden]

            if self.layer_wise == "ScalarMix":          
                    hiddens = self.layer_agg(hiddens, mask = attention_mask) # [1, batch, seq, hidden]
                    
        else:
            # else, uses only the last hidden states.
            hiddens = outputs.last_hidden_state # [batch, seq, hidden]
   
        # # predict with the first token
        if self.token_agg == 'sentence':
            hiddens = hiddens[:, 0, :] # [batch, 1, hidden]
        if self.token_agg == 'mean_token':
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(-1, -1, hiddens.size(-1)).float()
            
            # 1. Multiply by the mask to strictly zero-out all [PAD] tokens
            masked_embeddings = hiddens * input_mask_expanded
            
            # 2. Sum the valid embeddings along the sequence length (dim=1)
            sum_embeddings = torch.sum(masked_embeddings, dim=1)
            
            # 3. Sum the mask to find the true, unpadded length of each sequence
            # We use torch.clamp to mathematically prevent division by zero
            sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
            
            # 4. Divide the sum by the true sequence length to get the precise mean
            hiddens = sum_embeddings / sum_mask 
                
            # Final shape of hiddens: [batch_size, hidden_size]
        
        loss = None
        logits = self.regressor(hiddens)
    
        if labels is not None:
            loss_fct_mse = nn.MSELoss(reduction='mean')
            loss_fct_ce = nn.CrossEntropyLoss(reduction='mean')
            
            loss = loss_fct_mse(logits.view(-1), labels.view(-1))
        
        return CustomClassifierOutput(
            loss=loss,
            logits=logits,
            attentions=outputs.attentions
        )

In [ ]:
# Grid search helper functions and execution for CloseTrack_Predictor
import random
import pandas as pd

# Ensure deterministic behavior for reproducibility
torch.manual_seed(1234)
random.seed(1234)
np.random.seed(1234)

# 1) Dataset prep helper

def prepare_datasets_backbone(backbone, component_order, l1, batch_size):
    data_files = load_data_paths(DATA_DIR, l1, "finetune")
    hf_dataset = load_dataset("csv", data_files=data_files)

    tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=True)
    cols_to_merge = component_order.split("; ")
    sep_token = f" {tokenizer.sep_token} " if tokenizer.sep_token else " "

    preprocessed_ds = preprocess_dataset(hf_dataset, cols_to_merge, sep_token)

    tokenized_ds = preprocessed_ds.map(
        lambda x: tokenizer(x["input_text"], truncation=True, padding=False),
        batched=True,
        desc="Tokenizing input text"
    )

    pos_to_idx = {pos:i for i, pos in enumerate(pos_list)}
    tokenized_ds = tokenized_ds.map(
        lambda x: {"pos_label": [pos_to_idx[val] for val in x["en_target_pos"]]},
        batched=True,
        remove_columns=["en_target_pos", "L1", "input_text"],
        desc="Itemizing POS tagging"
    )

    return tokenized_ds


# 2) Training and evaluation helper

def train_and_eval_config(
    backbone,
    tokenized_ds,
    lr,
    wd,
    dropout,
    token_agg,
    batch_size,
    epoch,
    warmup_ratio,
):
    def model_init():
        return CloseTrack_Predictor.from_pretrained(
            backbone,
            num_labels=1,
            dropout=dropout,
            layer_wise='ScalarMix',
            token_agg=token_agg
        )

    training_args = TrainingArguments(
        output_dir=str(MODELS_DIR / "grid_search"),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        num_train_epochs=int(epoch),
        per_device_train_batch_size=int(batch_size),
        per_device_eval_batch_size=int(batch_size),
        learning_rate=float(lr),
        weight_decay=float(wd),
        warmup_ratio=float(warmup_ratio),
        load_best_model_at_end=True,
        metric_for_best_model="rmse",
        greater_is_better=False,
        report_to="none",
        seed=1234,
    )

    model = model_init()

    # separate parameter groups: scalar mix weights in second group to apply lr2/wd2
    # scalar_params = [p for n, p in model.named_parameters() if "scalar_weights" in n or "gamma" in n]
    # remaining_params = [p for n, p in model.named_parameters() if not ("scalar_weights" in n or "gamma" in n)]

    # optimizer = AdamW([
    #     {"params": remaining_params, "lr": lr1, "weight_decay": wd1},
    #     {"params": scalar_params, "lr": lr2, "weight_decay": wd2},
    # ])

    # num_training_steps = (len(tokenized_ds["train"]) // batch_size) * epoch
    # num_warmup_steps = int(num_training_steps * warmup_ratio)
    # scheduler = get_linear_schedule_with_warmup(
    #     optimizer,
    #     num_warmup_steps=num_warmup_steps,
    #     num_training_steps=num_training_steps
    # )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        tokenizer=AutoTokenizer.from_pretrained(backbone, use_fast=True),
        data_collator=DataCollatorWithPadding(AutoTokenizer.from_pretrained(backbone, use_fast=True)),
        # optimizers=(optimizer, scheduler),
        compute_metrics=compute_metrics,
    )

    train_state = trainer.train()
    eval_metrics = trainer.evaluate(tokenized_ds["validation"])

    # collect scalar mix internal weights
    state_dict = trainer.model.state_dict()
    scalar_weights = None
    gamma = None
    for k in state_dict:
        if "layer_agg.scalar_weights" in k:
            scalar_weights = state_dict[k].detach().cpu().numpy().tolist()
        if "layer_agg.gamma" in k:
            gamma = state_dict[k].item()

    run_result = {
        "lr": lr,
        "wd": wd,
        "dropout": dropout,
        "token_agg": token_agg,
        "epoch": epoch,
        "batch_size": batch_size,
        "train_loss": getattr(train_state, "training_loss", None),
        "eval_loss": eval_metrics.get('eval_loss'),
        "eval_rmse": eval_metrics.get("eval_rmse"),
        "eval_pearson": eval_metrics.get("eval_pearson"),
        "scalar_weights": scalar_weights,
        "gamma": gamma,
    }
    
    cleanup_trainer_memory(trainer)

    return run_result


# 3) Execute grid search

backbone = 'xlm-roberta-base'
component_order = "L1_source_word; L1_context; en_target_clue; en_target_word"
batch_size = 64
l1 = 'xx'
learning_rate_list1 = [5e-5, 4e-5, 3e-5, 2e-5, 1e-5]
weight_decay_list1 = [0.1,0.01,0.001]
# learning_rate_list2 = [2e-2]
# weight_decay_list2 = [0]
# temp_list = [2, 20]
dropout_list = [0.1,0.2,0.3,0.4]
token_agg_list = ['sentence', 'mean_token']
warmup_ratio = 0.1
epoch = 5

print("Preparing dataset... this takes a moment")
tokenized_ds = prepare_datasets_backbone(backbone, component_order, l1, batch_size)

results = []
for lr in learning_rate_list1:
    for wd in weight_decay_list1:
                    for dropout in dropout_list:
                        for token_agg in token_agg_list:
                            print(f"Running: lr={lr}, wd={wd}, dropout={dropout}, token_agg={token_agg}")
                            run_key = f"lr={lr}_wd1={wd}_dropout={dropout}_tokenagg={token_agg}"
                            try:
                                res = train_and_eval_config(
                                    backbone,
                                    tokenized_ds,
                                    lr,
                                    wd,
                                    dropout,
                                    token_agg,
                                    batch_size,
                                    epoch,
                                    warmup_ratio,
                                )
                                res["status"] = "success"
                                res["error"] = None
                            except Exception as e:
                                res = {
                                    "lr": lr,
                                    "wd": wd,
                                    "dropout": dropout,
                                    "token_agg": token_agg,
                                    "epoch": epoch,
                                    "batch_size": batch_size,
                                    "train_loss": None,
                                    "eval_loss": None,
                                    "eval_rmse": None,
                                    "eval_pearson": None,
                                    "scalar_weights": None,
                                    "gamma": None,
                                    "status": "failed",
                                    "error": str(e),
                                }
                                print(f"Run failed [{run_key}]: {e}")

                            results.append(res)
                            df = pd.DataFrame(results)
                            
                            df.to_csv("grid_search_results.csv", index=False)

print("Grid search done. Results written to grid_search_results.csv")


In [ ]:
lr = 1e-5

tokenized_ds = prepare_datasets_backbone(backbone, component_order, l1, batch_size)

def model_init():
    return CloseTrack_Predictor.from_pretrained(
        backbone,
        num_labels=1,
        dropout=dropout,
        layer_wise='ScalarMix',
        token_agg=token_agg
    )

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "grid_search"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=int(epoch),
    per_device_train_batch_size=int(batch_size),
    per_device_eval_batch_size=int(batch_size),
    learning_rate=float(lr),
    weight_decay=float(wd),
    warmup_ratio=float(warmup_ratio),
    load_best_model_at_end=True,
    metric_for_best_model="rmse",
    greater_is_better=False,
    report_to="none",
    seed=1234,
)

model = model_init()

# separate parameter groups: scalar mix weights in second group to apply lr2/wd2
# scalar_params = [p for n, p in model.named_parameters() if "scalar_weights" in n or "gamma" in n]
# remaining_params = [p for n, p in model.named_parameters() if not ("scalar_weights" in n or "gamma" in n)]

# optimizer = AdamW([
#     {"params": remaining_params, "lr": lr1, "weight_decay": wd1},
#     {"params": scalar_params, "lr": lr2, "weight_decay": wd2},
# ])

# num_training_steps = (len(tokenized_ds["train"]) // batch_size) * epoch
# num_warmup_steps = int(num_training_steps * warmup_ratio)
# scheduler = get_linear_schedule_with_warmup(
#     optimizer,
#     num_warmup_steps=num_warmup_steps,
#     num_training_steps=num_training_steps
# )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=AutoTokenizer.from_pretrained(backbone, use_fast=True),
    data_collator=DataCollatorWithPadding(AutoTokenizer.from_pretrained(backbone, use_fast=True)),
    # optimizers=(optimizer, scheduler),
    compute_metrics=compute_metrics,
)

train_state = trainer.train()
eval_metrics = trainer.evaluate(tokenized_ds["validation"])

# collect scalar mix internal weights
state_dict = trainer.model.state_dict()
scalar_weights = None
gamma = None
for k in state_dict:
    if "layer_agg.scalar_weights" in k:
        scalar_weights = state_dict[k].detach().cpu().numpy().tolist()
    if "layer_agg.gamma" in k:
        gamma = state_dict[k].item()

In [154]:
dropout = 0.3
token_agg = 'mean_token'

def model_init():
    return CloseTrack_Predictor.from_pretrained(
        backbone,
        num_labels=1,
        dropout=dropout,
        layer_wise='ScalarMix',
        token_agg=token_agg
    )

model = model_init()
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
single_batch = next(iter(DataLoader(tokenized_ds["train"], batch_size=8, collate_fn=DataCollatorWithPadding(AutoTokenizer.from_pretrained(backbone, use_fast=True)))))
single_batch = {k: v.to(model.device) for k, v in single_batch.items()}

for i in range(50):
    optimizer.zero_grad()
    outputs = model(**single_batch)
    loss = outputs.loss
    loss.backward()
    
    # Check for NaNs in gradients BEFORE stepping
    for name, param in model.named_parameters():
        if param.grad is not None and torch.isnan(param.grad).any():
            print(f"CRITICAL: NaN gradient detected in {name} at step {i}")
            break
            
    optimizer.step()
    print(f"Step {i} | Loss: {loss.item():.4f}")

Some weights of CloseTrack_Predictor were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['layer_agg.gamma', 'layer_agg.scalar_weights', 'regressor.0.bias', 'regressor.0.weight', 'regressor.1.denominator', 'regressor.1.numerator', 'regressor.3.bias', 'regressor.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 0 | Loss: 6.4885
Step 1 | Loss: 6.4886
Step 1 | Loss: 6.4886
Step 2 | Loss: 6.4886
Step 2 | Loss: 6.4886
Step 3 | Loss: 6.4885
Step 3 | Loss: 6.4885
Step 4 | Loss: 6.4886
Step 4 | Loss: 6.4886
Step 5 | Loss: 6.4885
Step 5 | Loss: 6.4885
Step 6 | Loss: 6.4885
Step 6 | Loss: 6.4885
Step 7 | Loss: 6.4886
Step 7 | Loss: 6.4886
Step 8 | Loss: 6.4886
Step 8 | Loss: 6.4886
Step 9 | Loss: 6.4885
Step 9 | Loss: 6.4885
Step 10 | Loss: 6.4884
Step 10 | Loss: 6.4884
Step 11 | Loss: 6.4884
Step 11 | Loss: 6.4884
Step 12 | Loss: 6.4884
Step 12 | Loss: 6.4884
Step 13 | Loss: 6.4885
Step 13 | Loss: 6.4885
Step 14 | Loss: 6.4883
Step 14 | Loss: 6.4883
Step 15 | Loss: 6.4884
Step 15 | Loss: 6.4884
Step 16 | Loss: 6.4883
Step 16 | Loss: 6.4883
Step 17 | Loss: 6.4884
Step 17 | Loss: 6.4884
Step 18 | Loss: 6.4883
Step 18 | Loss: 6.4883
Step 19 | Loss: 6.4882
Step 19 | Loss: 6.4882
Step 20 | Loss: 6.4883
Step 20 | Loss: 6.4883
Step 21 | Loss: 6.4881
Step 21 | Loss: 6.4881
Step 22 | Loss: 6.4882
Step 22

KeyboardInterrupt: 

In [106]:
eval_metrics

{'eval_loss': 3.2860660552978516,
 'eval_rmse': 1.8127509355545044,
 'eval_pearson': nan,
 'eval_runtime': 0.4921,
 'eval_samples_per_second': 4126.932,
 'eval_steps_per_second': 65.023,
 'epoch': 5.0}

In [107]:
model = trainer.model

In [147]:
data = next(iter(torch.utils.data.DataLoader(tokenized_ds['train'], batch_size=64, shuffle=True, collate_fn=data_collator)))

In [150]:
device = torch.device('cuda')

In [152]:
for n, v in model.named_parameters():
    if "layer_agg" in n:
        print(f"{n}: {v}")

layer_agg.scalar_weights: Parameter containing:
tensor([-1.3134e-36,  7.4703e-42,  9.1336e+17,  3.0749e-41,  9.4456e+22,
         4.5916e-41,  9.4455e+22,  4.5916e-41,  5.6052e-45,  0.0000e+00,
         3.3302e-11,  0.0000e+00,  9.4456e+22], device='cuda:0',
       requires_grad=True)
layer_agg.gamma: Parameter containing:
tensor([0.], device='cuda:0', requires_grad=True)


In [151]:
model(data['input_ids'].to(device), data['attention_mask'].to(device))

CustomClassifierOutput(loss=None, logits=tensor([[-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [-0.0210],
        [

In [ ]:
pd.set_option('display.max_rows', 100)
result = pd.read_csv('grid_search_results.csv')

In [103]:
result.sort_values('train_loss').iloc[0]

lr                                                           0.0002
wd                                                             0.01
dropout                                                         0.3
token_agg                                                mean_token
epoch                                                             5
batch_size                                                       64
train_loss                                                 1.902538
eval_rmse                                                       NaN
eval_pearson                                                    NaN
scalar_weights    [-0.07993323355913162, -0.0010982570238411427,...
gamma                                                      0.043872
status                                                      success
error                                                           NaN
Name: 13, dtype: object

In [99]:
import json
scalar_weights = torch.tensor(json.loads(result.sort_values('train_loss').iloc[0]['scalar_weights']))
F.softmax(scalar_weights, dim=0)

tensor([0.0769, 0.0832, 0.0851, 0.0873, 0.0856, 0.0849, 0.0838, 0.0824, 0.0825,
        0.0830, 0.0000, 0.0836, 0.0815])

In [102]:
gamma = torch.tensor(result.sort_values('train_loss').iloc[0]['gamma'])
gamma

tensor(0.0439, dtype=torch.float64)

In [ ]:
backbone = 'xlm-roberta-base'
component_order = "L1_source_word; L1_context; en_target_clue; en_target_word"
batch_size = 64
l1 = 'xx'
learning_rate_list1 = [2e-3,5e-3,2e-4,5e-4,2e-5,5e-5]
weight_decay_list1 = [0.1,0.01]
learning_rate_list2 = [2e-2, 5e-2, 2e-3, 5e-3]
weight_decay_list2 = [0, 0.1]
temp_list = [2, 20, 200]
dropout_list = [0.1, 0.2, 0.3]
token_agg_list = ['sentence', "mean_token"]
warmup_ratio = 0.1
epoch=5


temp = temp_list[0]
learning_rate1 = learning_rate_list1[0]
weight_decay1 = weight_decay_list1[0]
learning_rate2 = learning_rate_list2[0]
weight_decay2 = weight_decay_list2[0]
dropout = dropout_list[0]
token_agg = token_agg_list[0]

print(
    f"""temperature: {float(temp)}
        lr1: {float(learning_rate1)}
        weight_decay1: {float(weight_decay1)}
        lr2: {float(learning_rate2)}
        weight_decay2: {float(weight_decay2)}
        dropout: {float(dropout)}
        token_agg: {token_agg}
    """
)

def model_init_function():
        return CloseTrack_Predictor.from_pretrained(
            backbone, 
            num_labels=1,
            temp=temp,
            dropout=dropout,
            layer_wise='ScalarMix',
            token_agg=token_agg
        )

try:
    logging.info(f"Fine-tuning model...")

    # Load dataset paths and Hugging Face DatasetDict
    data_files = load_data_paths(DATA_DIR, l1, "finetune")
    hf_dataset = load_dataset("csv", data_files=data_files)

    # Load tokenizer and prepare input text formatting
    tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=True)
    cols_to_merge = component_order.split("; ")
    sep_token = f" {tokenizer.sep_token} " if tokenizer.sep_token else " "

    # Preprocess dataset: format input text, rename target label and remove extra columns
    preprocessed_ds = preprocess_dataset(hf_dataset, cols_to_merge, sep_token)

    # Tokenize dataset
    tokenized_ds = preprocessed_ds.map(
        lambda x: tokenizer(x["input_text"], truncation=True),
        batched=True,
        desc="Tokenizing input text"
    )

    # Itemize pos tagging 
    pos_to_idx = {pos:i for i, pos in enumerate(pos_list)}
    tokenized_ds = tokenized_ds.map(
        lambda x: {"pos_label": [pos_to_idx[val] for val in x['en_target_pos']]},
        batched=True,
        remove_columns=['en_target_pos', 'L1', 'input_text'],
        desc="Itemizing POS tagging"
    )

    # Set up training arguments
    training_args = TrainingArguments(
        # output_dir=str(MODELS_DIR / model_name),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        # save_total_limit=1,
        # max_grad_norm=1.0, # gradient clipping
        num_train_epochs=int(epoch),
        per_device_train_batch_size=int(batch_size),
        per_device_eval_batch_size=int(batch_size),
        learning_rate=float(learning_rate1),
        weight_decay=float(weight_decay1),
        warmup_ratio=float(warmup_ratio),
        load_best_model_at_end=True,
        report_to="none",
        seed=1234
    )

    # Initialise trainer
    data_collator = DataCollatorWithPadding(tokenizer)
    # CustomDataCollator(tokenizer, custom_features=["freq_feature", "pos_label", "label"])

    # set customized optimizer
    # 1. Initialize the model first so we can access its parameters
    model = model_init_function()
    custom_param_names = ["numerator", "denominator"] # [] 
    
    custom_params = [p for n, p in model.named_parameters() if any(nd in n for nd in custom_param_names)]
    base_params = [p for n, p in model.named_parameters() if not any(nd in n for nd in custom_param_names)]

    # 3. Create Custom Optimizer with separate rules
    optimizer = AdamW([
        {
            "params": base_params, 
            "lr": training_args.learning_rate, 
            "weight_decay": training_args.weight_decay
        },
        {
            "params": custom_params, 
            "lr": learning_rate2,             # Massive LR for the scalars
            "weight_decay": weight_decay2     # STRICTLY ZERO weight decay
        }
    ])

    # 4. Rebuild the learning rate scheduler (Trainer usually does this automatically, 
    # but we must do it manually since we are overriding the optimizer)
    num_training_steps = (len(tokenized_ds["train"]) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
    num_warmup_steps = int(num_training_steps * training_args.warmup_ratio)
    
    lr_scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    # Initialize the callback to watch your specific custom architecture weights
    debug_callback = OptimizerStateMonitor(
        params_to_watch=["scalar_weights", 'gamma'] 
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        data_collator=data_collator,
        optimizers=(optimizer, lr_scheduler),
        compute_metrics=compute_metrics,
        callbacks=[debug_callback]
    )
    
    trainer.train()
    
except Exception:
    logging.exception(f"Failed model")
    raise